In [0]:
print("Spark version:", spark.version)

Spark version: 4.1.0


In [0]:
teams_df = spark.read.csv(
    "s3://ipl-2010-data/Raw/teams_info.csv",
    header=True,
    inferSchema=True
)

display(teams_df)

team_name,url,espn_id
Mumbai Indians,https://assets.iplt20.com/ipl/MI/Logos/Logooutline/MIoutline.png,4346
Chennai Super Kings,https://assets.iplt20.com/ipl/CSK/logos/Logooutline/CSKoutline.png,4343
Delhi Capitals,https://assets.iplt20.com/ipl/DC/Logos/LogoOutline/DCoutline.png,4347
Kolkata Knight Respn_iders,https://assets.iplt20.com/ipl/KKR/Logos/Logooutline/KKRoutline.png,4341
Punjab Kings,https://assets.iplt20.com/ipl/PBKS/Logos/Logooutline/PBKSoutline.png,4342
Rajasthan Royals,https://assets.iplt20.com/ipl/RR/Logos/Logooutline/RRoutline.png,4345
Royal Challengers Bengaluru,https://assets.iplt20.com/ipl/RCB/Logos/Logooutline/RCBoutline.png,4340
Sunrisers Hyderabad,https://assets.iplt20.com/ipl/SRH/Logos/Logooutline/SRHoutline.png,5143
Pune Warriors,https://upload.wikimedia.org/wikipedia/en/4/4a/Pune_Warriors_India_IPL_Logo.png,4787
Kochi Tuskers Kerala,https://upload.wikimedia.org/wikipedia/en/thumb/9/96/Kochi_Tuskers_Kerala_Logo.svg/1280px-Kochi_Tuskers_Kerala_Logo.svg.png,4788


In [0]:
players_df = spark.read.csv(
    "s3://ipl-2010-data/Raw/2024_players_details.csv",
    header=True,
    inferSchema=True
)

matches_df = spark.read.csv(
    "s3://ipl-2010-data/Raw/Match_Info.csv",
    header=True,
    inferSchema=True
)

ball_df = spark.read.csv(
    "s3://ipl-2010-data/Raw/Ball_By_Ball_Match_Data.csv",
    header=True,
    inferSchema=True
)

print("Players:", players_df.count())
print("Matches:", matches_df.count())
print("Balls:", ball_df.count())
print("Teams:", teams_df.count())

Players: 261
Matches: 1243
Balls: 295732
Teams: 19


In [0]:
print("===== PLAYERS =====")
players_df.printSchema()

print("===== MATCHES =====")
matches_df.printSchema()

print("===== BALL BY BALL =====")
ball_df.printSchema()

print("===== TEAMS =====")
teams_df.printSchema()

===== PLAYERS =====
root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- longName: string (nullable = true)
 |-- battingName: string (nullable = true)
 |-- fieldingName: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- battingStyles: string (nullable = true)
 |-- longBattingStyles: string (nullable = true)
 |-- bowlingStyles: string (nullable = true)
 |-- longBowlingStyles: string (nullable = true)
 |-- playingRoles: string (nullable = true)
 |-- espn_url: string (nullable = true)

===== MATCHES =====
root
 |-- match_number: integer (nullable = true)
 |-- team1: string (nullable = true)
 |-- team2: string (nullable = true)
 |-- match_date: date (nullable = true)
 |-- toss_winner: string (nullable = true)
 |-- toss_decision: string (nullable = true)
 |-- result: string (nullable = true)
 |-- eliminator: string (nullable = true)
 |-- winner: string (nullable = true)
 |-- player_of_match: string (nullable = tr

In [0]:
print("Match IDs:")
matches_df.select("match_number").show(10, truncate=False)

print("Ball IDs:")
ball_df.select("ID").distinct().show(10, truncate=False)

Match IDs:
+------------+
|match_number|
+------------+
|501208      |
|1216505     |
|1359481     |
|1178398     |
|1529313     |
|1304095     |
|1178404     |
|980979      |
|1529311     |
|734045      |
+------------+
only showing top 10 rows
Ball IDs:
+-------+
|ID     |
+-------+
|1178398|
|501244 |
|1082646|
|1216505|
|1178405|
|336029 |
|980979 |
|1175369|
|1359481|
|1178404|
+-------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import col

silver_ball_df = (
    ball_df.alias("b")
    .join(
        matches_df.alias("m"),
        col("b.ID") == col("m.match_number"),
        "left"
    )
    .select(
        col("b.ID").alias("match_id"),
        col("m.match_date"),
        col("m.team1"),
        col("m.team2"),
        col("m.toss_winner"),
        col("m.toss_decision"),
        col("m.winner"),
        col("m.player_of_match"),
        col("m.venue"),
        col("m.city"),
        col("b.Innings").alias("innings"),
        col("b.Overs").alias("over"),
        col("b.BallNumber").alias("ball_number"),
        col("b.Batter").alias("batter"),
        col("b.Bowler").alias("bowler"),
        col("b.NonStriker").alias("non_striker"),
        col("b.ExtraType").alias("extra_type"),
        col("b.BatsmanRun").alias("batsman_runs"),
        col("b.ExtrasRun").alias("extras_runs"),
        col("b.TotalRun").alias("total_runs"),
        col("b.IsWicketDelivery").alias("is_wicket"),
        col("b.PlayerOut").alias("player_out"),
        col("b.Kind").alias("dismissal_kind"),
        col("b.BattingTeam").alias("batting_team")
    )
)

display(silver_ball_df.limit(10))

match_id,match_date,team1,team2,toss_winner,toss_decision,winner,player_of_match,venue,city,innings,over,ball_number,batter,bowler,non_striker,extra_type,batsman_runs,extras_runs,total_runs,is_wicket,player_out,dismissal_kind,batting_team
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,1,S Sohal,Z Khan,S Dhawan,null,0,0,0,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,2,S Sohal,Z Khan,S Dhawan,null,1,0,1,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,3,S Dhawan,Z Khan,S Sohal,null,1,0,1,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,4,S Sohal,Z Khan,S Dhawan,null,0,0,0,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,5,S Sohal,Z Khan,S Dhawan,wides,0,1,1,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,6,S Sohal,Z Khan,S Dhawan,null,0,0,0,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,0,7,S Sohal,Z Khan,S Dhawan,null,0,0,0,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,1,1,S Dhawan,JJ van der Wath,S Sohal,wides,0,1,1,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,1,2,S Dhawan,JJ van der Wath,S Sohal,noballs,0,1,1,0,NA,NA,Deccan Chargers
501208,2011-04-14,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Deccan Chargers,DW Steyn,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,1,1,3,S Dhawan,JJ van der Wath,S Sohal,null,4,0,4,0,NA,NA,Deccan Chargers


In [0]:
from pyspark.sql.functions import col

silver_ball_df = (
    ball_df.alias("b")
    .join(
        matches_df.alias("m"),
        col("b.ID") == col("m.match_number"),
        "left"
    )
    .select(
        col("b.ID").alias("match_id"),
        col("m.match_date"),
        col("m.team1"),
        col("m.team2"),
        col("m.toss_winner"),
        col("m.toss_decision"),
        col("m.winner"),
        col("m.player_of_match"),
        col("m.venue"),
        col("m.city"),
        col("b.Innings").alias("innings"),
        col("b.Overs").alias("over"),
        col("b.BallNumber").alias("ball_number"),
        col("b.Batter").alias("batter"),
        col("b.Bowler").alias("bowler"),
        col("b.NonStriker").alias("non_striker"),
        col("b.ExtraType").alias("extra_type"),
        col("b.BatsmanRun").alias("batsman_runs"),
        col("b.ExtrasRun").alias("extras_runs"),
        col("b.TotalRun").alias("total_runs"),
        col("b.IsWicketDelivery").alias("is_wicket"),
        col("b.PlayerOut").alias("player_out"),
        col("b.Kind").alias("dismissal_kind"),
        col("b.BattingTeam").alias("batting_team")
    )
)

print("Silver DataFrame created:", silver_ball_df.count())

Silver DataFrame created: 295732


In [0]:
silver_path = "s3://ipl-2010-data/Silver/ball_by_ball/"

silver_ball_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver data written successfully!")

Silver data written successfully!


In [0]:
silver_check_df = spark.read.parquet(
    "s3://ipl-2010-data/Silver/ball_by_ball/"
)

print("Silver rows:", silver_check_df.count())
display(silver_check_df.limit(10))

Silver rows: 295732


match_id,match_date,team1,team2,toss_winner,toss_decision,winner,player_of_match,venue,city,innings,over,ball_number,batter,bowler,non_striker,extra_type,batsman_runs,extras_runs,total_runs,is_wicket,player_out,dismissal_kind,batting_team
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,3,7,V Kohli,Avesh Khan,D Padikkal,null,0,0,0,1,V Kohli,bowled,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,1,D Padikkal,I Sharma,RM Patidar,null,0,0,0,1,D Padikkal,bowled,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,2,GJ Maxwell,I Sharma,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,3,GJ Maxwell,I Sharma,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,4,GJ Maxwell,I Sharma,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,5,GJ Maxwell,I Sharma,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,4,6,GJ Maxwell,I Sharma,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,5,1,RM Patidar,Avesh Khan,GJ Maxwell,null,1,0,1,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,5,2,GJ Maxwell,Avesh Khan,RM Patidar,null,4,0,4,0,NA,NA,Royal Challengers Bangalore
1254079,2021-04-27,Royal Challengers Bangalore,Delhi Capitals,Delhi Capitals,field,Royal Challengers Bangalore,AB de Villiers,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,1,5,3,GJ Maxwell,Avesh Khan,RM Patidar,null,0,0,0,0,NA,NA,Royal Challengers Bangalore


In [0]:
silver_check_df.createOrReplaceTempView("ipl_deliveries")

print("SQL view created!")

SQL view created!


In [0]:
%sql
SELECT
    batter,
    SUM(batsman_runs) AS total_runs,
    COUNT(DISTINCT match_id) AS matches_played
FROM ipl_deliveries
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 10;

batter,total_runs,matches_played
V Kohli,9346,275
RG Sharma,7331,275
S Dhawan,6769,221
DA Warner,6567,184
KL Rahul,5828,149
SK Raina,5536,200
MS Dhoni,5439,241
AM Rahane,5367,197
SV Samson,5181,185
AB de Villiers,5181,170


In [0]:
gold_top_batsmen = spark.sql("""
    SELECT
        batter,
        SUM(batsman_runs) AS total_runs,
        COUNT(DISTINCT match_id) AS matches_played
    FROM ipl_deliveries
    GROUP BY batter
    ORDER BY total_runs DESC
    LIMIT 10
""")

gold_top_batsmen.write \
    .mode("overwrite") \
    .parquet("s3://ipl-2010-data/Gold/top_batsmen/")

print("Gold table saved successfully!")

Gold table saved successfully!


In [0]:
%sql
SELECT
    bowler,
    COUNT(*) AS wickets
FROM ipl_deliveries
WHERE is_wicket = 1
  AND dismissal_kind NOT IN ('run out', 'retired hurt', 'obstructing the field')
GROUP BY bowler
ORDER BY wickets DESC
LIMIT 10;

bowler,wickets
YS Chahal,233
B Kumar,226
SP Narine,209
PP Chawla,192
JJ Bumrah,190
R Ashwin,187
DJ Bravo,183
RA Jadeja,180
Rashid Khan,179
A Mishra,174


In [0]:
gold_top_bowlers = spark.sql("""
    SELECT
        bowler,
        COUNT(*) AS wickets
    FROM ipl_deliveries
    WHERE is_wicket = 1
      AND dismissal_kind NOT IN (
          'run out',
          'retired hurt',
          'obstructing the field'
      )
    GROUP BY bowler
    ORDER BY wickets DESC
    LIMIT 10
""")

gold_top_bowlers.write \
    .mode("overwrite") \
    .parquet("s3://ipl-2010-data/Gold/top_bowlers/")

print("Gold top bowlers table saved successfully!")

Gold top bowlers table saved successfully!


In [0]:
%sql
SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) AS wins,
    ROUND(
        SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) * 100.0
        / COUNT(*),
        2
    ) AS win_percentage
FROM (
    SELECT match_id, team1 AS team, winner
    FROM ipl_deliveries
    GROUP BY match_id, team1, winner

    UNION ALL

    SELECT match_id, team2 AS team, winner
    FROM ipl_deliveries
    GROUP BY match_id, team2, winner
)
GROUP BY team
ORDER BY win_percentage DESC;

team,matches_played,wins,win_percentage
Royal Challengers Bengaluru,46,29,63.04
Rising Pune Supergiant,16,10,62.50
Gujarat Titans,77,47,61.04
Chennai Super Kings,266,148,55.64
Mumbai Indians,291,155,53.26
Kolkata Knight Riders,278,140,50.36
Rajasthan Royals,251,123,49.00
Sunrisers Hyderabad,211,102,48.34
Delhi Capitals,120,58,48.33
Royal Challengers Bangalore,240,114,47.50


In [0]:
gold_team_performance = spark.sql("""
    SELECT
        team,
        COUNT(*) AS matches_played,
        SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) AS wins,
        ROUND(
            SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS win_percentage
    FROM (
        SELECT match_id, team1 AS team, winner
        FROM ipl_deliveries
        GROUP BY match_id, team1, winner

        UNION ALL

        SELECT match_id, team2 AS team, winner
        FROM ipl_deliveries
        GROUP BY match_id, team2, winner
    )
    GROUP BY team
    ORDER BY win_percentage DESC
""")

gold_team_performance.write \
    .mode("overwrite") \
    .parquet("s3://ipl-2010-data/Gold/team_performance/")

print("Gold team performance saved successfully!")

Gold team performance saved successfully!


In [0]:
%sql
SELECT
    toss_decision,
    COUNT(*) AS matches,
    SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS toss_winner_wins,
    ROUND(
        SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) * 100.0
        / COUNT(*),
        2
    ) AS win_percentage
FROM (
    SELECT DISTINCT
        match_id,
        toss_winner,
        toss_decision,
        winner
    FROM ipl_deliveries
)
GROUP BY toss_decision
ORDER BY win_percentage DESC;

toss_decision,matches,toss_winner_wins,win_percentage
field,825,443,53.70
bat,418,185,44.26


In [0]:
gold_toss_analysis = spark.sql("""
    SELECT
        toss_decision,
        COUNT(*) AS matches,
        SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS toss_winner_wins,
        ROUND(
            SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS win_percentage
    FROM (
        SELECT DISTINCT
            match_id,
            toss_winner,
            toss_decision,
            winner
        FROM ipl_deliveries
    )
    GROUP BY toss_decision
    ORDER BY win_percentage DESC
""")

display(gold_toss_analysis)

toss_decision,matches,toss_winner_wins,win_percentage
field,825,443,53.70
bat,418,185,44.26


In [0]:
gold_toss_analysis.write \
    .mode("overwrite") \
    .parquet("s3://ipl-2010-data/Gold/toss_analysis/")

print("Gold toss analysis saved successfully!")

Gold toss analysis saved successfully!


In [0]:
gold_batsmen = spark.read.parquet(
    "s3://ipl-2010-data/Gold/top_batsmen/"
)

gold_bowlers = spark.read.parquet(
    "s3://ipl-2010-data/Gold/top_bowlers/"
)

gold_teams = spark.read.parquet(
    "s3://ipl-2010-data/Gold/team_performance/"
)

gold_toss = spark.read.parquet(
    "s3://ipl-2010-data/Gold/toss_analysis/"
)

gold_batsmen.createOrReplaceTempView("gold_batsmen")
gold_bowlers.createOrReplaceTempView("gold_bowlers")
gold_teams.createOrReplaceTempView("gold_teams")
gold_toss.createOrReplaceTempView("gold_toss")

print("Gold SQL views created!")

Gold SQL views created!


In [0]:
%sql
SELECT
    batter,
    total_runs,
    matches_played
FROM gold_batsmen
ORDER BY total_runs DESC
LIMIT 10;


batter,total_runs,matches_played
V Kohli,9346,275
RG Sharma,7331,275
S Dhawan,6769,221
DA Warner,6567,184
KL Rahul,5828,149
SK Raina,5536,200
MS Dhoni,5439,241
AM Rahane,5367,197
SV Samson,5181,185
AB de Villiers,5181,170


In [0]:
%sql
SELECT
    bowler,
    wickets
FROM gold_bowlers
ORDER BY wickets DESC
LIMIT 10;

bowler,wickets
YS Chahal,233
B Kumar,226
SP Narine,209
PP Chawla,192
JJ Bumrah,190
R Ashwin,187
DJ Bravo,183
RA Jadeja,180
Rashid Khan,179
A Mishra,174


In [0]:
%sql
SELECT * FROM gold_batsmen ORDER BY total_runs DESC;
SELECT * FROM gold_bowlers ORDER BY wickets DESC;
SELECT * FROM gold_teams ORDER BY win_percentage DESC;
SELECT * FROM gold_toss;

toss_decision,matches,toss_winner_wins,win_percentage
field,825,443,53.70
bat,418,185,44.26


In [0]:
gold_batsmen.toPandas().to_csv("/tmp/top_batsmen.csv", index=False)
gold_bowlers.toPandas().to_csv("/tmp/top_bowlers.csv", index=False)
gold_teams.toPandas().to_csv("/tmp/team_performance.csv", index=False)
gold_toss.toPandas().to_csv("/tmp/toss_analysis.csv", index=False)

print("All Gold datasets exported!")

All Gold datasets exported!


In [0]:
print("Batsmen:", gold_batsmen.count())
print("Bowlers:", gold_bowlers.count())
print("Teams:", gold_teams.count())
print("Toss:", gold_toss.count())

Batsmen: 10
Bowlers: 10
Teams: 19
Toss: 2
